In [1]:
import duckdb

def esql(sql, conn=None):
    if conn is None:
        conn = duckdb.connect()

    return conn.execute(sql).df()


# Titanic

In [2]:
import seaborn as sns
import pandas as pd

titanic = pd.read_csv("seaborn_titanic.csv")
titanic.head()

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True


survived: 생존 여부 (0 사망, 1 생존)
pclass: 객실 등급 (1=상, 2=중, 3=하)
sex: 성별
age: 나이
sibsp: 함께 탑승한 형제/자매/배우자 수
parch: 함께 탑승한 부모/자녀 수
fare: 운임 요금
embarked: 탑승 항구 (C Cherbourg, Q Queenstown, S Southampton)
class: pclass의 문자열 버전 (First, Second, Third)
who: 사람 구분 (man, woman, child)
adult_male: 성인 남성 여부 (True/False)
deck: 선실 갑판 구역 (A~G, 결측 많음)
embark_town: 탑승 항구 이름(문자열)
alive: 생존 여부 문자열 (yes/no)
alone: 단독 탑승 여부 (True/False)

In [3]:
print(titanic.shape)
print(titanic.isna().sum())

(891, 15)
survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64


In [4]:
# Titanic 전처리 예시
# - 중복/누수 컬럼 정리
# - 결측치 처리
# - 파생변수 생성

import numpy as np
import pandas as pd

eda_df = titanic.copy()

# 1) 누수/중복 성격 컬럼 제거
# alive: survived와 동일 의미(yes/no)
# class: pclass와 중복 성격
# embark_town: embarked와 중복 성격
eda_df = eda_df.drop(columns=["alive", "class", "embark_town"])

# 2) 결측치 처리
# age는 성별+객실등급 그룹 중앙값으로 보정, 남은 결측은 전체 중앙값
eda_df["age"] = eda_df.groupby(["sex", "pclass"])["age"].transform(lambda s: s.fillna(s.median()))
eda_df["age"] = eda_df["age"].fillna(eda_df["age"].median())

# embarked는 최빈값으로 보정
eda_df["embarked"] = eda_df["embarked"].fillna(eda_df["embarked"].mode().iloc[0])

# deck는 결측이 매우 많아 제거
eda_df = eda_df.drop(columns=["deck"])

# 3) 파생변수
eda_df["family_size"] = eda_df["sibsp"] + eda_df["parch"] + 1

print("shape:", eda_df.shape)
print("missing:\n", eda_df.isna().sum())
eda_df.head()

shape: (891, 12)
missing:
 survived       0
pclass         0
sex            0
age            0
sibsp          0
parch          0
fare           0
embarked       0
who            0
adult_male     0
alone          0
family_size    0
dtype: int64


,survived,pclass,sex,age,sibsp,parch,fare,embarked,who,adult_male,alone,family_size
0,0,3,male,22.0,1,0,7.2500,S,man,True,False,2
1,1,1,female,38.0,1,0,71.2833,C,woman,False,False,2
2,1,3,female,26.0,0,0,7.9250,S,woman,False,True,1
3,1,1,female,35.0,1,0,53.1000,S,woman,False,False,2
4,0,3,male,35.0,0,0,8.0500,S,man,True,True,1


In [5]:
import duckdb
def esql(sql, conn=None):
    if conn is None:
        conn = duckdb.connect()

    return conn.execute(sql).df()

In [6]:
esql("select survived, count(*) from eda_df group by survived")

,survived,count_star()
0,1,342
1,0,549


In [7]:
esql("select  pclass, survived,count(*) from eda_df group by pclass,survived order by pclass")

,pclass,survived,count_star()
0,1,1,136
1,1,0,80
2,2,0,97
3,2,1,87
4,3,0,372
5,3,1,119


In [8]:
esql("select pclass, avg(survived), count(*) cnt from eda_df group by pclass order by pclass")

,pclass,avg(survived),cnt
0,1,0.629630,216
1,2,0.472826,184
2,3,0.242363,491


In [9]:
# 다른 변수중 survived한 사람을 예측하는데 도움이 되는 변수는?

In [10]:
esql("select survived, avg(age) from eda_df group by survived")


,survived,avg(age)
0,0,29.737705
1,1,28.108684


In [11]:
query="""
select case when age<10 then '1. <10' 
when age<20 then '2. 10~20' 
when age<30 then '3. 20~30' 
when age<40 then '4. 30~40' 
when age<50 then '5. 40~50' 
when age<60 then '6. 50~60' else 
'7. >60' end as age_group, 
avg(survived) survival_rate, count(*) cnt 
from eda_df group by age_group
order by age_group"""

esql(query)

,age_group,survival_rate,cnt
0,1. <10,0.612903,62
1,2. 10~20,0.401961,102
2,3. 20~30,0.315642,358
3,4. 30~40,0.454054,185
4,5. 40~50,0.354545,110
5,6. 50~60,0.416667,48
6,7. >60,0.269231,26


In [12]:
esql("select survived, min(fare), avg(fare), median(fare),max(fare), count(*) cnt from eda_df group by survived")


,survived,min(fare),avg(fare),median(fare),max(fare),cnt
0,1,0.0,48.395408,26.0,512.3292,342
1,0,0.0,22.117887,10.5,263.0000,549


In [13]:
esql("select age,fare,survived from eda_df where fare=0")

,age,fare,survived
0,36.0,0.0,0
1,40.0,0.0,0
2,25.0,0.0,1
3,30.0,0.0,0
4,19.0,0.0,0
5,30.0,0.0,0
6,30.0,0.0,0
7,30.0,0.0,0
8,49.0,0.0,0
9,40.0,0.0,0


In [14]:
esql("select survived, avg(sibsp) from eda_df group by survived")


,survived,avg(sibsp)
0,1,0.473684
1,0,0.553734


In [15]:
esql("select survived, avg(parch) from eda_df group by survived")


,survived,avg(parch)
0,0,0.329690
1,1,0.464912


In [16]:
esql("select sex, avg(survived) from eda_df group by sex")


,sex,avg(survived)
0,female,0.742038
1,male,0.188908


# Boston

In [17]:
import pandas as pd

boston = pd.read_csv("openml_boston_v1.csv")

# feature / target
X = boston.drop(columns="MEDV")
y = boston["MEDV"]

In [18]:
boston.head()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0,0.469,6.421,78.9,4.9671,2,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0,0.458,6.998,45.8,6.0622,3,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2


In [19]:
esql("select min(medv), avg(medv), median(medv), max(medv) from boston")

,min(medv),avg(medv),median(medv),max(medv)
0,5.0,22.532806,21.2,50.0


In [20]:
query="""select case when medv<10 then '1. <10' 
when medv<20 then '2. 10~20' 
when medv<30 then '3. 20~30' 
when medv<40 then '4. 30~40' 
when medv<60 then '5. 40~50' 
end as medv_group, 
avg(crim), avg(zn), avg(indus), avg(nox), avg(rm), avg(age), 
avg(dis), avg(tax), avg(ptratio), avg(b), avg(lstat), avg(medv), count(*) cnt
from boston 
group by medv_group order by medv_group
"""
esql(query)

,medv_group,avg(crim),avg(zn),avg(indus),avg(nox),avg(rm),avg(age),avg(dis),avg(tax),avg(ptratio),avg(b),avg(lstat),avg(medv),cnt
0,1. <10,22.059312,0.000000,18.903333,0.691167,5.728042,94.570833,1.699088,669.750000,20.191667,231.185417,26.359583,7.750000,24
1,2. 10~20,5.476324,3.051075,14.542849,0.617171,5.940113,85.277419,3.062532,486.231183,19.394086,328.586720,17.554140,16.024194,186
2,3. 20~30,1.050858,12.384434,9.498585,0.508335,6.250165,55.752830,4.452888,351.127358,18.287264,382.957075,9.778585,23.472170,212
3,4. 30~40,0.177886,35.141509,3.640189,0.471300,7.086491,50.816981,4.934392,277.207547,16.424528,391.150943,5.711321,33.735849,53
4,5. 40~50,1.555269,22.403226,8.707419,0.533806,7.647484,66.280645,3.366045,352.387097,16.103226,383.663871,4.160968,47.451613,31


# Diabetes

In [24]:
from types import SimpleNamespace

import pandas as pd

# 로컬 CSV에서 데이터 읽기
df = pd.read_csv("diabetes.csv")
df.head()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646,151.0
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204,75.0
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930,141.0
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362,206.0
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641,135.0


In [25]:
df.describe()

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6,target
count,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,4.420000e+02,442.000000
mean,-7.284269e-18,2.348549e-17,-2.087320e-16,-4.571507e-17,-9.293722e-18,4.420798e-17,2.135044e-18,2.913707e-17,9.143013e-17,1.431736e-17,152.133484
std,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,4.761905e-02,77.093005
min,-1.072256e-01,-4.464164e-02,-9.027530e-02,-1.123988e-01,-1.267807e-01,-1.156131e-01,-1.023071e-01,-7.639450e-02,-1.260971e-01,-1.377672e-01,25.000000
25%,-3.729927e-02,-4.464164e-02,-3.422907e-02,-3.665608e-02,-3.424784e-02,-3.035840e-02,-3.511716e-02,-3.949338e-02,-3.324559e-02,-3.317903e-02,87.000000
50%,5.383060e-03,-4.464164e-02,-7.283766e-03,-5.670422e-03,-4.320866e-03,-3.819065e-03,-6.584468e-03,-2.592262e-03,-1.947171e-03,-1.077698e-03,140.500000
75%,3.807591e-02,5.068012e-02,3.124802e-02,3.564379e-02,2.835801e-02,2.984439e-02,2.931150e-02,3.430886e-02,3.243232e-02,2.791705e-02,211.500000
max,1.107267e-01,5.068012e-02,1.705552e-01,1.320436e-01,1.539137e-01,1.987880e-01,1.811791e-01,1.852344e-01,1.335973e-01,1.356118e-01,346.000000


In [ ]:
# target과 연관성이 높은 변수는?